start_date='2025-04-01'
end_date='2025-04-30'

In [1]:
adhoc: str

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import json
from pyspark.sql.functions import current_timestamp, trunc, add_months, date_format, col
import pytz
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta # added this for datetime calculation
from pyspark.sql.functions import col, explode,round as sround
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from pyspark.sql.functions import col, lit, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType , DateType , BooleanType , DoubleType ,TimestampType,ArrayType,ArrayType,LongType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType
from pyspark.sql.utils import *
spark = SparkSession.builder.appName("json-to-parquet").config("spark.driver.maxResultSize", "-1").getOrCreate()
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import last_day
import pandas as pd
#import numpy as np
import calendar
from delta.tables import DeltaTable

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 4, Finished, Available, Finished, False)

In [3]:
# Get current time in Eastern Time
eastern = pytz.timezone("US/Eastern")
now_et = datetime.now(eastern)
#now_et = now_et - relativedelta(months=2)

# First day of this month (exclusive upper bound) -- to ensure end date be first of current month and hence avoiding missing edge cases 
end_date_prev_month = now_et.replace(day=1) 

# First day of previous month -- to ensure start date be first of previous month and hence avoiding missing edge cases
start_date_prev_month = end_date_prev_month - relativedelta(months=1)

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 5, Finished, Available, Finished, False)

In [4]:
eastern = pytz.timezone("US/Eastern")
current_datetime=datetime.now(eastern)
# Split into date and time
rundate = current_datetime.date()
runtime = current_datetime.time()

print("Run date:", rundate)
print("Run time:", runtime)

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 6, Finished, Available, Finished, False)

Run date: 2026-06-29
Run time: 13:58:37.421833


In [5]:
adhoc = 'No'

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 7, Finished, Available, Finished, False)

In [6]:
if adhoc == 'No':
    start_date = start_date_prev_month.strftime('%Y-%m-%d') # Format as string to ensure proper SQL comparision
    end_date = end_date_prev_month.strftime('%Y-%m-%d') # Format as string to ensure proper SQL comparision
else:
    start_date = now_et.replace(day=1).strftime('%Y-%m-%d')
    end_date=str(rundate)
print(start_date) # if not Adhoc should look like 2026-02-01
print(end_date) # if not Adhoc should look like 2026-03-01

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 8, Finished, Available, Finished, False)

2026-05-01
2026-06-01


In [7]:
Commissions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/AgentCommission_versions/'
PolicyCovPrem = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Risks_Vehicles_PremiumFactors_versions/'
PolicyTransactions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Transaction_versions/'
Policy_versions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Policy_versions'
policyfees = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/FeesAndTaxes_versions/'

financial = spark.sql("select * from PROD_DEC_IH.financial")
financial_type = spark.sql("select * from PROD_DEC_IH.financial_type")
general_codes = spark.sql("select * from PROD_DEC_IH.general_codes")
Coverage_codes = spark.sql("select * from PROD_DEC_IH.Coverage_codes")
Pr1_STATEMENT_NEXT_FEE_Q = spark.sql("select * from PROD_DEBIH.PR1_STATEMENT_NEXT_FEE_Q")
Pr1_STATEMENT_FEE = spark.sql("select * from PROD_DEBIH.PR1_STATEMENT_FEE")
pr1_accountp = spark.sql("select * from PROD_DEBIH.PR1_ACCOUNTP")

# Read data into DataFrames
Commissions = spark.read.format("delta").load(Commissions)
PolicyCovPrem = spark.read.format("delta").load(PolicyCovPrem)
PolicyTransactions=spark.read.format("delta").load(PolicyTransactions)
Policy_versions=spark.read.format("delta").load(Policy_versions)
policyfees=spark.read.format("delta").load(policyfees)


# Create temporary views
Commissions.createOrReplaceTempView("Commissions")
financial.createOrReplaceTempView("financial")
financial_type.createOrReplaceTempView("financial_type")
general_codes.createOrReplaceTempView("general_codes")
Coverage_codes.createOrReplaceTempView("Coverage_codes")
PolicyCovPrem.createOrReplaceTempView("PolicyCovPrem")
PolicyTransactions.createOrReplaceTempView("PolicyTransactions")
Policy_versions.createOrReplaceTempView("Policy_versions")
policyfees.createOrReplaceTempView("policyfees")
Pr1_STATEMENT_NEXT_FEE_Q.createOrReplaceTempView("Pr1_STATEMENT_NEXT_FEE_Q")
Pr1_STATEMENT_FEE.createOrReplaceTempView("Pr1_STATEMENT_FEE")
pr1_accountp.createOrReplaceTempView("pr1_accountp")

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 9, Finished, Available, Finished, False)

Query Architecture:

Commission Data
      │
Policy Transactions
      │
Premium Calculations
      │
Per-Day Allocation
      │
Earned / Unearned
      │
Claims Data
      │
Fees
      │
Union Everything
      │
Final Aggregated Report

In [8]:
query = f'''SELECT SF_FEE_CODE, SUM(SF_AMOUNT) 
FROM PR1_STATEMENT_FEE 
WHERE SF_PAID_DATE >= '{start_date}' AND SF_PAID_DATE < '{end_date}'
GROUP BY 1'''
spark.sql(query).show(20)	

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 10, Finished, Available, Finished, False)

+-----------+--------------+
|SF_FEE_CODE|sum(SF_AMOUNT)|
+-----------+--------------+
|         26|      10077.15|
|         27|        388.19|
|         22|      61347.06|
|          1|      92703.00|
|          3|       6240.00|
|          5|      36105.00|
|          4|       4640.00|
|          2|       1120.00|
+-----------+--------------+



In [9]:
def get_monthly_data(start: str, end: str, dr: str): #Function to get monthly data for given date range 
	query = f'''
With 

-- ## Commission Data ##
-- ## Pull only NEW transaction commissions where commission value > 0 
-- ## This creates a lookup table used later when calculating commission

agent_comm as 
(select distinct policy_ref,value,commissiontype
from Commissions where value>0 and transactiontype='NEW'
),

-- ## Latest Policy ##
-- ## Retrieves the most recent committed transaction per policy 
-- ## using row_number() ordered by transaction Date and Number used later for inforce prem Calculation

policy_latest as (
 select '40' as company,SUBSTRING(policynumber,1,2) as policy_state,t.Policy_ref,p.PolicyNumber,t.Type,p.PolicyStatus,p.EffectiveDate,p.ExpirationDate,
 row_number() over(partition by PolicyNumber order by Date desc,Number desc) as rn
from Policy_versions p
join PolicyTransactions t
on t.Policy_ref=p.Policy_ref
where t.Status='Committed'
AND t.Date < '{end}'
--and p.PolicyNumber in ('GAPA007708444','GAPA007709230')
),
policy_final_status AS (
    SELECT 
        PolicyNumber, 
        PolicyStatus
    FROM policy_latest 
    WHERE rn = 1 -- This ensures we get the most recent status (e.g., 'Policy Cancelled')
),
policy_data as(

-- ## Block 1 ##
-- ## Liability type coverages 
-- ## CommissionType = LBO 

select policynumber,cov.name as coverage,
'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
sum(cov.EffectivePremium) as netwritten,
SUM(CASE WHEN t.type <> 'Cancellation' 
	THEN cov.EffectivePremium 
	ELSE 0 
	END )as grosswritten_by_policy_cov,
SUM(IFNULL(cov.EffectivePremium*0.01*ag.Value,0)) as commission

from Policy_versions p

inner join PolicyTransactions t
on t.policy_ref=p.policy_ref

inner join PolicyCovPrem cov
on cov.policy_ref=t.policy_ref
and cov.Name in ('BI','PD','UMBI','UMPD','MP','FFH')

left join agent_comm ag
on ag.policy_ref=cov.policy_ref
and ag.CommissionType in ('LBO')

WHERE t.status = 'Committed' -- ## Considering only Committed Transactions ##
AND (
    -- ## Block 1:Transaction Date is after EffectiveDate and within range ##
    (t.Date > t.EffectiveDate AND t.Date >= '{start}' AND t.Date < '{end}')
    OR 
    -- ## Block 2: EffectiveDate is in range and Date hasn't passed the end ##
    (t.EffectiveDate >= '{start}' AND t.EffectiveDate < '{end}' AND t.Date < '{end}')
) -- ## Additional Date comparision block in line with business expectation ##
--and p.policynumber in ('GAPA007708444','GAPA007709230')

group by policynumber,cov.name

union

-- ## Block 2 ##
-- ## Physical damage coverages 
-- ## CommissionType = LBP 

select policynumber,cov.name as coverage,
'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
sum(cov.EffectivePremium) as netwritten,
SUM(CASE WHEN t.type <> 'Cancellation' 
	THEN cov.EffectivePremium 
	ELSE 0 
	END )as grosswritten_by_policy_cov,
SUM(IFNULL(cov.EffectivePremium*0.01*ag.Value,0)) as commission

from Policy_versions p

inner join PolicyTransactions t
on t.policy_ref=p.policy_ref

inner join PolicyCovPrem cov
on cov.policy_ref=t.policy_ref
and cov.Name in ('CMP','COL','TOW','RNT')

left join agent_comm ag
on ag.policy_ref=cov.policy_ref
and ag.CommissionType in ('LBP')

WHERE t.status = 'Committed' -- ## Considering only Committed Transactions ##
AND (
    -- ## Block 1: Date is after EffectiveDate and within range ##
    (t.Date > t.EffectiveDate AND t.Date >= '{start}' AND t.Date < '{end}')
    OR 
    -- ## Block 2: EffectiveDate is in range and Date hasn't passed the end ##
    (t.EffectiveDate >= '{start}' AND t.EffectiveDate < '{end}' AND t.Date < '{end}')
) --## Additional Date comparision block added in line with business expectation ##
--and p.policynumber in ('GAPA007708444','GAPA007709230')

group by policynumber,cov.name

union

-- ## Block 3 ##
-- ## Handles other coverages where commission type equals coverage name
-- ## Excludes LBO and LBP which were handled above

select policynumber,cov.name as coverage,
'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
0.00 as netwritten,
0.00 as grosswritten_by_policy_cov,
sum(IFNULL(cov.EffectivePremium*0.01*ag.Value,0)) as commission

from Policy_versions p

inner join PolicyTransactions t
on t.policy_ref=p.policy_ref

inner join PolicyCovPrem cov
on cov.policy_ref=t.policy_ref

inner join agent_comm ag
on ag.policy_ref=cov.policy_ref
and ag.CommissionType=cov.name
and ag.CommissionType not in ('LBO','LBP')

WHERE t.status = 'Committed' -- ## Considering only Committed Transactions ##
AND (
    -- ## Block 1: Date is after EffectiveDate and within range ##
    (t.Date > t.EffectiveDate AND t.Date >= '{start}' AND t.Date < '{end}')
    OR 
    -- ## Block 2: EffectiveDate is in range and Date hasn't passed the end ##
    (t.EffectiveDate >= '{start}' AND t.EffectiveDate < '{end}' AND t.Date < '{end}')
) -- ## Additional Date comparision block added in line with business expectation ##
group by policynumber,cov.name
) 
,

-- ## Build policy transaction windows ##
-- ## TransactionEndDate ##
-- ## Determined using LEAD() to find next transaction effective date
-- ## If none exists, use policy expiration date

transactiondate As (
SELECT 
    p.Policy_ref,
    PolicyNumber , 
    p.EffectiveDate ,
    p.ExpirationDate ,
    PolicyStatus, 
    DATEDIFF(DAY,cast(p.EffectiveDate as DATE),cast(p.ExpirationDate as DATE)) as PolicyTerm, 
	cast(t.EffectiveDate as DATE) as TransactionEffectiveDate,
    CASE
        WHEN LEAD(CAST(t.EffectiveDate AS DATE)) OVER (PARTITION BY PolicyNumber order by CAST(t.EffectiveDate AS DATE), t.Number ) is not NULL 
		THEN cast(LEAD(CAST(t.EffectiveDate AS DATE)) OVER (PARTITION BY PolicyNumber order by CAST(t.EffectiveDate AS DATE), t.Number ) as DATE)
        ELSE cast(ExpirationDate as DATE)
    END AS TransactionEndDate,
    t.Type,
	cast(t.Date as DATE) Date,
    Number
    FROM Policy_versions p
    
	JOIN PolicyTransactions t 
	on t.Policy_ref=p.Policy_ref  
    
	where  t.status='Committed' 
    AND t.Date < '{end}' --## Added this condition to ensure cosistency with date condition ##
	--and t.date between '{start}' and '{end}'
	--and p.policynumber in ('GAPA007708444','GAPA007709230')
) 
,

-- ## Convert annual premium into per-day premium so we can calculate: Earned Premium, Unearned Premium ##

per_day as(  
SELECT 
dt.Policy_ref,
dt.PolicyNumber , 
dt.EffectiveDate as PolicyEffDate,
dt.ExpirationDate as PolicyExpirationDate,
dt.PolicyStatus, 
dt.PolicyTerm,
dt.TransactionEffectiveDate,
--convert_timezone(dt.TransactionEndDate,('UTC'),('EST')) as TransactionEndDate,
--dt.type as TransactionType,
dt.number,

covprem.UnitId as VehicleId,   
covprem.Name as CoverageName,
covprem.AnnualPremium as CoverageAnnualPremium ,
covprem.EffectivePremium as CoverageEffectivePremium ,
(TRY_CAST(covprem.AnnualPremium as DECIMAL(18,6))/dt.PolicyTerm) as covPerDayPremium,

-- ## EffectiveDays ##
-- ## Determines how many days within the reporting window the coverage was active

 CASE
    WHEN dt.Type ='Cancellation' THEN 0 
    WHEN TransactionEffectiveDate < dt.Date and TransactionEndDate <= dt.Date and dt.Date between '{start}' and  '{end}'
            THEN DATEDIFF(DAY,TransactionEffectiveDate,TransactionEndDate)
	WHEN TransactionEffectiveDate < dt.Date and TransactionEndDate <= dt.Date and dt.Date < '{start}' 
			THEN 0 -- ## Added missing condition to Cover all cases ##		
    WHEN TransactionEffectiveDate < dt.Date and TransactionEndDate > dt.Date and TransactionEndDate between  '{start}' and   '{end}'
            THEN DATEDIFF(DAY,TransactionEffectiveDate, TransactionEndDate) -- ## Added missing condition to Cover all cases ##
	WHEN TransactionEffectiveDate < dt.Date and TransactionEffectiveDate <  '{start}' AND dt.Date <  '{start}' AND TransactionEndDate >=   '{end}'
         THEN DATEDIFF(DAY, TransactionEffectiveDate , '{end}')	 		
    WHEN TransactionEffectiveDate >= dt.Date AND dt.TransactionEndDate >=   '{end}'AND TransactionEffectiveDate <=  '{start}' Then DATEDIFF(DAY, '{start}',  '{end}') 
    WHEN TransactionEffectiveDate >= dt.Date AND dt.TransactionEndDate <   '{end}'AND TransactionEffectiveDate <=  '{start}' THEN DATEDIFF(DAY, '{start}',dt.TransactionEndDate)
    WHEN TransactionEffectiveDate >= dt.Date AND dt.TransactionEndDate >=   '{end}'AND TransactionEffectiveDate >  '{start}' Then DATEDIFF(DAY,TransactionEffectiveDate,  '{end}')
    WHEN TransactionEffectiveDate >= dt.Date AND dt.TransactionEndDate <   '{end}'AND TransactionEffectiveDate >  '{start}' THEN DATEDIFF(DAY,TransactionEffectiveDate,dt.TransactionEndDate)
	ELSE DATEDIFF(DAY, TransactionEffectiveDate , '{end}') -- ## Added Else Case to add a fallback option ##
END as EffectiveDays, 
DATEDIFF(DAY,'{end}',cast(dt.ExpirationDate as DATE)) as UnearnedDays,
rank() over(partition by PolicyNumber,covprem.Name order by TransactionEffectiveDate desc,Number desc) as rnk

from transactiondate dt

JOIN PolicyCovPrem covprem 
on covprem.Policy_ref=dt.Policy_ref

where dt.Date < '{end}' -- ## Comparing with transaction Date instead of Transaction Effective Date earlier ##

),

-- ## Calculate earned premium and unearned premium per coverage per policy within the date range ##
earned_prem as
(
	
select '40' as company,
'PA' as policy_type,
SUBSTRING(PolicyNumber,1,2) as policy_state,
PolicyNumber,
SUM(CASE
    WHEN PolicyStatus ='Policy Cancelled' THEN 0
    WHEN EffectiveDays<0 THEN 0
    ELSE ROUND(EffectiveDays * TRY_CAST(covPerDayPremium AS DECIMAL(18,6)),2) END )AS EarnedPrem,

SUM(CASE WHEN rnk=1 THEN
						CASE WHEN PolicyStatus ='Policy Cancelled' THEN 0
							 WHEN UnearnedDays<0 THEN 0
							 ELSE ROUND(UnearnedDays * TRY_CAST(covPerDayPremium AS DECIMAL(18,6)),2) END
			ELSE 0 END
		) AS UnearnedPrem,

	CoverageName as coverage

from per_day
group by CoverageName,PolicyNumber
)
,

-- ## Calculate: Incurred Loss, Paid Loss ##
claims_data as(
select finc_clmc_cov_code,
finc_clm_policy_pgm, 
finc_clm_lob, 
finc_clm_policy_st,
	IFNULL(SUM(
							CASE 
							   WHEN fint_categ = 'R'  THEN finc_amnt
                               WHEN fint_categ = 'C'  THEN -finc_amnt

                               ELSE 0
                           END
						   ), 0) +
               IFNULL(SUM((CASE 
								WHEN fint_categ = 'V'  THEN finc_amnt
								WHEN fint_categ = 'N'  THEN -finc_amnt
                                ELSE 0
                           END)), 0) +
               IFNULL(SUM((CASE 
								WHEN fint_categ = '%'  THEN finc_amnt
								WHEN fint_categ = '^'  THEN -finc_amnt
                                ELSE 0
                           END)), 0) -

		         -- subtracting salvage, subro and loss recoveries from incurred loss
               IFNULL(SUM((CASE 
								WHEN fint_categ = '3' THEN -finc_amnt
								WHEN fint_categ = 'S' THEN finc_amnt
                                ELSE 0
                           END)), 0) -
               IFNULL(SUM((CASE 
								WHEN fint_categ = '4' THEN -finc_amnt
								WHEN fint_categ = 'X' THEN finc_amnt
                                ELSE 0
                           END)), 0) -

				IFNULL(SUM((CASE 
								WHEN fint_categ = '5' THEN -finc_amnt
								WHEN fint_categ = 'Y' THEN finc_amnt
								ELSE 0
						   END)), 0) as Incurred,


			IFNULL(SUM((
               CASE 
			        WHEN fint_categ = 'P' THEN finc_amnt
                    WHEN fint_categ = '1' THEN -finc_amnt
                    ELSE 0
               END
           )), 0)
		   +
		   	IFNULL(SUM(CASE 
		 			WHEN fint_categ = 'E' THEN finc_amnt
                    WHEN fint_categ = '2' THEN -finc_amnt
                    ELSE 0
                 END), 0) -
		  --subtracting recoveries from paid loss
            IFNULL(SUM((CASE 
					WHEN fint_categ = '3' THEN -finc_amnt
					WHEN fint_categ = 'S' THEN finc_amnt
                             ELSE 0
                        END)), 0) -
            IFNULL(SUM((CASE 
					WHEN fint_categ = '4'  THEN -finc_amnt
					WHEN fint_categ = 'X'  THEN finc_amnt
                             ELSE 0
                        END)), 0) - 
            IFNULL(SUM((CASE 
								WHEN fint_categ = '5' THEN -finc_amnt
								WHEN fint_categ = 'Y' THEN finc_amnt
                                ELSE 0
                           END)), 0)
			AS paid_loss


 FROM 
        financial f  
		inner join (select distinct policynumber from policy_data)p
		on p.policynumber=f.finc_policy    
        INNER JOIN financial_type ft ON f.finc_opr_type = ft.fint_type
        INNER JOIN general_codes g ON g.gcp_code = f.finc_clm_policy_pgm
--		inner join Coverage_codes cov on f.finc_clmc_cov_code=cov.cov_code
    WHERE
	 (f.finc_acc_date >= '{start}' and f.finc_acc_date < '{end}') --## Between being replaced with current logic to avoid missing on edge case transactions ##
		AND gcp_internal_type = 'COMPANY' 
        AND gcp_valid_code = '1'
        AND gcp_valid_type = '1'
        AND f.finc_sup_valid = '1'
        AND f.finc_rev_trans_flg = '0'
        AND (f.finc_opr_approval_status = 'A' OR f.finc_opr_approval_status IS NULL)
		--and finc_claim_nbr in ('2000002652','2000002653','2000002654','2000002656')
    GROUP BY 
		f.finc_clmc_cov_code, 
        f.finc_clm_policy_pgm, 
        f.finc_clm_lob,
		f.finc_clm_policy_st
),

-- ## Captures 3 types of fees:  'Policy Fee', 'Blling Fee'(Late fee, NSF fee,Reinstatement fee) , 'Statement Fee' (Installment fee, EFT fee) ## 

fees as
(
select policynumber,
'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
CASE WHEN f.code = 'NSDTRAVELCLUB' THEN 'ADD' else f.code end as fee_code,
f.description as fee_desc,
sum(amount) as written_fees,
0.00 as written_prem
from 
policy_versions p

join PolicyTransactions t
on t.policy_ref=p.policy_ref
and t.status='Committed'
and t.date >= '{start}' and t.date <'{end}' --## Between being replaced with current logic to avoid missing on edge case transactions ##

join policyfees f
on f.policy_ref=t.policy_ref

where f.code not in ('INSFEE')
--and p.policynumber in ('GAPA007708444','GAPA007709230')

group by policynumber, f.code,f.description

union all

-- ## billing fees (late fee, nsf fee, reinstate fee) ##
select p.accountp_policy_num as policynumber,
'40' as company,'PA' as policy_type,SUBSTRING(p.accountp_policy_num,1,2) as policy_state,
CASE SFQ_FEE_CODE WHEN 5 THEN 'LAT'
				  WHEN 2 THEN 'NSF'
				  WHEN 4 THEN 'RNS' 
	END	as fee_code,
CASE SFQ_FEE_CODE WHEN 5 THEN 'Late Fee'
				  WHEN 2 THEN 'Non-Sufficient Funds Fee'
				  WHEN 4 THEN 'Reinstatement Fee' 
	END as fee_desc,
SUM(fq.SFQ_FEE_AMOUNT) as written_fees,
0.00 as written_prem

from PR1_STATEMENT_NEXT_FEE_Q fq
join pr1_accountp p
on p.accountp_number=fq.SFQ_APPLY_TO_ACCOUNT
where SFQ_FEE_CODE in(5,2,4)
and p.accountp_policy_condition=1
and SFQ_IS_VALID = 1
and SFQ_IS_WAIVED <> 1
and fq.SFQ_PAID_DATE >= '{start}' and fq.SFQ_PAID_DATE < '{end}' --## Between being replaced with current logic to avoid missing on edge case transactions ##
--and p.accountp_number in ('GAPA007708444','GAPA007709230')

group by p.accountp_policy_num,SFQ_FEE_CODE

union all

-- ## statement fees (installment and eft fees) ##

select p.accountp_policy_num as policynumber,
'40' as company,'PA' as policy_type,SUBSTRING(p.accountp_policy_num,1,2) as policy_state,
CASE SF_FEE_CODE WHEN 1 THEN 'INSFEE'
				 WHEN 3 THEN 'EFT'
	END	as fee_code,
CASE SF_FEE_CODE WHEN 1 THEN 'Installment Fee'
				 WHEN 3 THEN 'EFT Fee' 
	END as fee_desc,
SUM(SF_AMOUNT) as written_fees,
0.00 as written_prem

from PR1_STATEMENT_FEE f
join pr1_accountp p
on p.accountp_number=f.SF_STATEMENT_NUMBER

where SF_FEE_CODE in(1,3)
and p.accountp_policy_condition=1
and SF_IS_REVERSED <> 1 
and SF_IS_WAIVED <> 1
and SF_IS_VALID = 1
and SF_PAID_DATE >='{start}' and SF_PAID_DATE < '{end}' --## Between being replaced with current logic to avoid missing on edge case transactions ##
--and p.accountp_number in ('GAPA007708444','GAPA007709230')

group by p.accountp_policy_num,SF_FEE_CODE

),

-- ## This step combines all components into a single dataset: 
-- ## policy_data	written premium
-- ## policy_latest	inforce premium
-- ## claims_data	losses
-- ## earned_prem	earned/unearned
-- ## fees	fee income
-- ## Each block fills only relevant columns and sets others to 0

union_data as
(
select p.company as company,
policy_type,
policy_state,
CASE WHEN p.coverage = 'UMBI' THEN 'UMB'
	 WHEN p.coverage = 'UMPD' THEN 'UMP'
--	 WHEN p.coverage in ('BI','PD') THEN 'LIA'
	 ELSE p.coverage END as coverage,
sum(grosswritten_by_policy_cov) as grosswritten,
sum(grosswritten_by_policy_cov) - sum(netwritten) as cancelled,
0.00 as fees,
sum(netwritten) as netwritten,
sum(commission) as commission,
0.00 as EarnedPrem,
0.00 as UnearnedPrem,
0.00 as Incurred,
0.00 as paid,
0.00 as inforce_prem,
0 as inforce_policies

from policy_data p

group by p.company,
policy_type,
policy_state,
CASE WHEN p.coverage = 'UMBI' THEN 'UMB'
	 WHEN p.coverage = 'UMPD' THEN 'UMP'
--	 WHEN p.coverage in ('BI','PD') THEN 'LIA'
	 ELSE p.coverage END

union all

select company,
'PA' as policy_type,
policy_state,
CASE WHEN cov.name = 'UMBI' THEN 'UMB'
	 WHEN cov.name = 'UMPD' THEN 'UMP'
--	 WHEN cov.name in ('BI','PD') THEN 'LIA'
	 ELSE cov.name END as coverage,
0.00 as grosswritten,
0.00 as cancelled,
0.00 as fees,
0.00 as netwritten,
0.00 as commission,
0.00 as EarnedPrem,
0.00 as UnearnedPrem,
0.00 as Incurred,
0.00 as paid,
sum (cov.annualPremium) as inforce_prem,
count(Distinct p.PolicyNumber) as inforce_policies 

from policy_latest p

inner join PolicyCovPrem cov
on cov.policy_ref=p.policy_ref

where rn=1 
and p.Type <> 'Cancellation' 
and p.ExpirationDate >= '{end}' --## Not considering Expiration Date equal to 1st of current month  ##

group by company,
policy_state,
CASE WHEN cov.name = 'UMBI' THEN 'UMB'
	 WHEN cov.name = 'UMPD' THEN 'UMP'
--	 WHEN cov.name in ('BI','PD') THEN 'LIA'
	 ELSE cov.name END


union all

select finc_clm_policy_pgm as company,
CASE WHEN clm.finc_clm_lob = 'PPA' THEN 'PA' ELSE clm.finc_clm_lob END as policy_type,
clm.finc_clm_policy_st as policy_state,
 CASE WHEN cov.cov_stat_code like '[0-9]' THEN cov.cov_code
	  --WHEN cov.cov_stat_code ='LIA' and cov.cov_code in ('BI','PD') THEN cov.cov_stat_code
	  WHEN cov.cov_stat_code ='LIA' THEN cov.cov_code
	  ELSE cov.cov_stat_code
	END AS coverage,
0.00 as grosswritten,
0.00 as cancelled,
0.00 as fees,
0.00 as netwritten,
0.00 as commission,
0.00 as EarnedPrem,
0.00 as UnearnedPrem,
SUM(Incurred) as Incurred,
SUM(paid_loss) as paid,
0.00 as inforce_prem,
0 as inforce_policies

from claims_data clm
join Coverage_codes cov
on clm.finc_clmc_cov_code = cov.cov_code

group by finc_clm_policy_pgm,finc_clm_lob, finc_clm_policy_st, 
 CASE WHEN cov.cov_stat_code like '[0-9]' THEN cov.cov_code
	  --WHEN cov.cov_stat_code ='LIA' and cov.cov_code in ('BI','PD') THEN cov.cov_stat_code
	  WHEN cov.cov_stat_code ='LIA' THEN cov.cov_code
	  ELSE cov.cov_stat_code END

union all

select company,
policy_type,
policy_state,
CASE WHEN coverage = 'UMBI' THEN 'UMB'
	 WHEN coverage = 'UMPD' THEN 'UMP'
	 --WHEN coverage in ('BI','PD') THEN 'LIA'
	 ELSE coverage END as coverage ,
0.00 as grosswritten,
0.00 as cancelled,
0.00 as fees,
0.00 as netwritten,
0.00 as commission,
sum(EarnedPrem) as EarnedPrem,
sum(UnearnedPrem) as UnearnedPrem,
0.00 as Incurred,
0.00 as paid,
0.00 as inforce_prem,
0 as inforce_policies

from earned_prem

group by company,
policy_type,
policy_state,
CASE WHEN coverage = 'UMBI' THEN 'UMB'
	 WHEN coverage = 'UMPD' THEN 'UMP'
	 --WHEN coverage in ('BI','PD') THEN 'LIA'
	 ELSE coverage END

union all

select company as company,
policy_type,
policy_state,
fee_code as coverage,
0.00 as grosswritten,
0.00 as cancelled,
sum(written_fees) as fees,
0.00 as netwritten,
0.00 as commission,
0.00 as EarnedPrem,
0.00 as UnearnedPrem,
0.00 as Incurred,
0.00 as paid,
0.00 as inforce_prem,
0 as inforce_policies

from fees
group by company,policy_type,policy_state,fee_code

) select '{dr}' AS date_range,
concat(date_format(cast('{start_date}' as date),'MMMM'),' ',YEAR('{start_date}')) as reporting_period, --## Reporting Period to use start_date instead of end_date ##
date_format('{rundate}','MM/dd/yyyy') as rundate,
SUBSTRING('{runtime}',1,8) as runtime,
gcomp.gcp_desc as company,
CONCAT(policy_type,' - ',glob.gcp_desc) as policy_type,
policy_state,
un.coverage,
COALESCE(COALESCE(gcov.gcp_desc,cov.cov_desc),f.fee_desc) as coverage_desc,
sum(grosswritten) as grosswritten,
sum(cancelled) as cancelled,
sum(netwritten) as netwritten,
CASE WHEN {dr} = '2' or {dr} = '3' THEN 0.00 ELSE sum(fees) END as fees,
sum(commission) as commission,
sum(EarnedPrem) as EarnedPrem,
CASE WHEN {dr} = '2' or {dr} = '3' THEN 0.00 ELSE sum(UnearnedPrem) END as UnearnedPrem,
sum(Incurred) as Incurred,
sum(paid) as paid,
CASE WHEN sum(grosswritten)=0 THEN 0 ELSE sum(paid)/sum(grosswritten) end as loss_ratio,
CASE WHEN sum(netwritten)=0 THEN 0 ELSE (sum(commission)*100)/sum(netwritten) end as avg_commission,
CASE WHEN {dr} = '2' or {dr} = '3' THEN 0.00 ELSE sum(inforce_prem) END as inforce_prem,
CASE WHEN {dr} = '2' or {dr} = '3' THEN 0.00 ELSE max(inforce_policies) END as inforce_policies

from union_data un

left join general_codes gcomp
on gcp_code=un.company
and gcomp.gcp_internal_type='COMPANY'

left join Coverage_codes cov
on un.coverage = cov.cov_code

left join general_codes gcov
on gcov.gcp_code=un.coverage
and gcov.gcp_internal_type='GROUP_COVERAGE'

left join general_codes glob
on glob.gcp_code = CASE WHEN un.policy_type = 'PA' THEN 'PPA' ELSE un.policy_type END
and glob.gcp_internal_type='LOB'

left join 
(select distinct fee_code,fee_desc from fees) f
on f.fee_code=un.coverage

group by gcomp.gcp_desc,CONCAT(policy_type,' - ',glob.gcp_desc),policy_state, un.coverage
		,COALESCE(COALESCE(gcov.gcp_desc,cov.cov_desc),f.fee_desc)

 '''
	return spark.sql(query)	


StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 11, Finished, Available, Finished, False)

In [10]:
result = get_monthly_data(start_date, end_date,1)
result.show(100)
result = result.replace(float('nan'), None)
result.write.mode('overwrite').format('delta').option("overwriteSchema", "true").save('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/SR23217A_policy_breakdown')

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 12, Finished, Available, Finished, False)

+----------+----------------+----------+--------+----------------+------------------+------------+----------+--------------------+------------------+------------------+------------------+------------------+------------------+----------+------------+--------+----+----------+------------------+------------+----------------+
|date_range|reporting_period|   rundate| runtime|         company|       policy_type|policy_state|  coverage|       coverage_desc|      grosswritten|         cancelled|        netwritten|              fees|        commission|EarnedPrem|UnearnedPrem|Incurred|paid|loss_ratio|    avg_commission|inforce_prem|inforce_policies|
+----------+----------------+----------+--------+----------------+------------------+------------+----------+--------------------+------------------+------------------+------------------+------------------+------------------+----------+------------+--------+----+----------+------------------+------------+----------------+
|         1|        May 2026

In [11]:
#In line with keepting our start_date to first of month and end date to start of next month 
def get_month_start(year, month):
    return datetime(year, month, 1).strftime("%Y-%m-%d")

def get_month_end(year, month):
    return (datetime(year, month, 1) + relativedelta(months=1)).strftime("%Y-%m-%d")

def format_reporting_period(year, month):
    return datetime(year, month, 1).strftime('%B %Y')

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 13, Finished, Available, Finished, False)

In [12]:
now = datetime.today()

n=2

periods_to_refresh = [
    (now - relativedelta(months=i)).strftime("%B %Y")
    for i in range(0, n)
]

print(periods_to_refresh)

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 14, Finished, Available, Finished, False)

['October 2025']


In [13]:
period_list = ",".join([f"'{p}'" for p in periods_to_refresh])

spark.sql(f'''
DELETE FROM SR23217A_1
WHERE reporting_period IN ({period_list})
''')

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 15, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]

In [14]:
# New Code for enabling Multiple month Functionality
# Current date
now = datetime.today()

df_final = None  # Master DataFrame

# Loop from current month back to 6 months ago (total 9 iterations) -- can be increased at cost of performance 
for i in range(0,n): 
    rpt_date = now - relativedelta(months=i)
    rpt_year = rpt_date.year
    rpt_month = rpt_date.month

    reporting_period = format_reporting_period(rpt_year, rpt_month)

    # ------------------
    # DR = 1 → Current Month
    start_curr = get_month_start(rpt_year, rpt_month)
    end_curr = get_month_end(rpt_year, rpt_month)
    df1 = get_monthly_data(start_curr, end_curr, 1)
    df1 = df1.withColumn("reporting_period", lit(reporting_period)).withColumn("dr", lit(1))

    # DR = 2 → Year-To-Date
    start_ytd = get_month_start(rpt_year, 1)
    end_ytd = get_month_end(rpt_year, rpt_month)
    df2 = get_monthly_data(start_ytd, end_ytd, 2)
    df2 = df2.withColumn("reporting_period", lit(reporting_period)).withColumn("dr", lit(2))

    # DR = 3 → Last 13 Months (ending at current iteration month)
    df3_combined = None
    for j in range(13):
        temp_date = rpt_date - relativedelta(months=j)
        start_13 = get_month_start(temp_date.year, temp_date.month)
        end_13 = get_month_end(temp_date.year, temp_date.month)
        df_month = get_monthly_data(start_13, end_13, 3)
        df_month = df_month.withColumn("reporting_period", lit(reporting_period)).withColumn("dr", lit(3))
        df3_combined = df_month if df3_combined is None else df3_combined.unionByName(df_month)

    # Combine all 3 dr types for this reporting period
    df_combined = df1.unionByName(df2).unionByName(df3_combined)

    # Append to final dataset
    df_final = df_combined if df_final is None else df_final.unionByName(df_combined)

# Final result
df_final.show(50)

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 16, Finished, Available, Finished, False)

+----------+----------------+----------+--------+----------------+------------------+------------+----------+--------------------+------------------+------------------+------------------+------------------+------------------+----------+------------+--------+----+----------+------------------+------------+----------------+---+
|date_range|reporting_period|   rundate| runtime|         company|       policy_type|policy_state|  coverage|       coverage_desc|      grosswritten|         cancelled|        netwritten|              fees|        commission|EarnedPrem|UnearnedPrem|Incurred|paid|loss_ratio|    avg_commission|inforce_prem|inforce_policies| dr|
+----------+----------------+----------+--------+----------------+------------------+------------+----------+--------------------+------------------+------------------+------------------+------------------+------------------+----------+------------+--------+----+----------+------------------+------------+----------------+---+
|         1|    

In [15]:
df_final.selectExpr(
    "min(to_date(reporting_period,'MMMM yyyy')) as min_period","max(to_date(reporting_period,'MMMM yyyy')) as max_period"  
).show()

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 17, Finished, Available, Finished, False)

+----------+----------+
|min_period|max_period|
+----------+----------+
|2025-10-01|2025-10-01|
+----------+----------+



In [16]:
df2 = (
    df_final
    .withColumn("grosswritten", sround(col("grosswritten").cast("double"), 2))
    .withColumn("cancelled", sround(col("cancelled").cast("double"), 2))
    .withColumn("netwritten", sround(col("netwritten").cast("double"), 2))
    .withColumn("fees", sround(col("fees").cast("double"), 2))
    .withColumn("commission", sround(col("commission").cast("double"), 2))
    .withColumn("EarnedPrem", sround(col("EarnedPrem").cast("double"), 2))
    .withColumn("UnearnedPrem", sround(col("UnearnedPrem").cast("double"), 2))
    .withColumn("Incurred", sround(col("Incurred").cast("double"), 2))
    .withColumn("paid", sround(col("paid").cast("double"), 2))
    .withColumn("loss_ratio", sround(col("loss_ratio").cast("double"), 1))
    .withColumn("avg_commission", sround(col("avg_commission").cast("double"), 1))
    .withColumn("inforce_prem", sround(col("inforce_prem").cast("double"), 2))
    .withColumn("inforce_policies", sround(col("inforce_policies").cast("int"), 2))
)

# single conversion to pandas
df = df2.toPandas()

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 18, Finished, Available, Finished, False)

In [17]:
schema  = StructType([
StructField("date_range", StringType(), True),
StructField("reporting_period", StringType(), True),
StructField("rundate",StringType(), True),
StructField("runtime",StringType(), True),
StructField("company", StringType(), True),
StructField("policy_type", StringType(), True),
StructField("policy_state", StringType(), True),
StructField("coverage", StringType(), True),
StructField("coverage_desc", StringType(), True),
StructField("grosswritten", DoubleType(), True),
StructField("cancelled", DoubleType(), True),
StructField("netwritten", DoubleType(), True),
StructField("fees", DoubleType(), True),
StructField("commission", DoubleType(), True),
StructField("EarnedPrem", DoubleType(), True),
StructField("UnearnedPrem", DoubleType(), True),
StructField("Incurred", DoubleType(), True),
StructField("paid", DoubleType(), True),
StructField("loss_ratio", DoubleType(), True),
StructField("avg_commission", DoubleType(), True),
StructField("inforce_prem", DoubleType(), True),
StructField("inforce_policies", IntegerType(), True),
])  
try:
    df_spark=spark.createDataFrame(df,schema)
except:

    df_spark=spark.createDataFrame([],schema)
df_spark.show()    

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 19, Finished, Available, Finished, False)

+----------+----------------+----------+--------+----------------+------------------+------------+----------+--------------------+------------+---------+----------+--------+----------+----------+------------+--------+----+----------+--------------+------------+----------------+
|date_range|reporting_period|   rundate| runtime|         company|       policy_type|policy_state|  coverage|       coverage_desc|grosswritten|cancelled|netwritten|    fees|commission|EarnedPrem|UnearnedPrem|Incurred|paid|loss_ratio|avg_commission|inforce_prem|inforce_policies|
+----------+----------------+----------+--------+----------------+------------------+------------+----------+--------------------+------------+---------+----------+--------+----------+----------+------------+--------+----+----------+--------------+------------+----------------+
|         1|    October 2025|06/29/2026|13:58:37|Southern General|PA - Personal Auto|          GA|       NSF|Non-Sufficient Fu...|         0.0|      0.0|       0.0

In [18]:
df_spark.write.mode('append').format('delta').option("overwriteSchema", "true").save('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/SR23217A_1')

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 20, Finished, Available, Finished, False)

In [19]:
# ── Reconciliation Check: SR23217A_1 vs Data Extract ─────────────────────────
CHECK_PERIOD = 'April 2026'

# ── Step 1: Check what columns exist ─────────────────────────────────────────
print("── Columns in SR23217A_1 ──")
spark.sql("SELECT * FROM SR23217A_1 LIMIT 1").printSchema()

# ── Step 2: Check what periods exist ─────────────────────────────────────────
print("\n── All reporting periods in SR23217A_1 ──")
spark.sql("""
    SELECT 
        reporting_period,
        date_range,
        COUNT(*) AS row_count
    FROM SR23217A_1
    GROUP BY reporting_period, date_range
    ORDER BY to_date(reporting_period, 'MMMM yyyy'), date_range
""").show(50, truncate=False)

# ── Step 3: Current Month totals (date_range = '1') ──────────────────────────
print(f"\n── Current Month (date_range=1) for {CHECK_PERIOD} ──")
spark.sql(f"""
    SELECT
        date_range,
        CASE date_range
            WHEN '1' THEN 'Current Month'
            WHEN '2' THEN 'Year-To-Date'
            WHEN '3' THEN 'Most Recent 13 Months'
        END                                      AS period_label,
        ROUND(SUM(grosswritten), 2)              AS Gross_Written,
        ROUND(SUM(cancelled), 2)                 AS Cancelled_Returned,
        ROUND(SUM(netwritten), 2)                AS Net_Written,
        ROUND(SUM(fees), 2)                      AS Fees,
        ROUND(SUM(EarnedPrem), 2)                AS Earned_Prem,
        ROUND(SUM(UnearnedPrem), 2)              AS Unearned_Prem,
        ROUND(SUM(Incurred), 2)                  AS Incurred_Losses,
        ROUND(SUM(commission), 2)                AS Commission,
        ROUND(SUM(inforce_prem), 2)              AS Inforce_Prem
    FROM SR23217A_1
    WHERE reporting_period = '{CHECK_PERIOD}'
    GROUP BY date_range
    ORDER BY date_range
""").show(truncate=False)

StatementMeta(, ba58f95a-1f81-4224-9716-fe83a1ea0cec, 21, Finished, Available, Finished, False)

── Columns in SR23217A_1 ──
root
 |-- date_range: string (nullable = true)
 |-- reporting_period: string (nullable = true)
 |-- rundate: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- company: string (nullable = true)
 |-- policy_type: string (nullable = true)
 |-- policy_state: string (nullable = true)
 |-- coverage: string (nullable = true)
 |-- coverage_desc: string (nullable = true)
 |-- grosswritten: double (nullable = true)
 |-- cancelled: double (nullable = true)
 |-- netwritten: double (nullable = true)
 |-- fees: double (nullable = true)
 |-- commission: double (nullable = true)
 |-- EarnedPrem: double (nullable = true)
 |-- UnearnedPrem: double (nullable = true)
 |-- Incurred: double (nullable = true)
 |-- paid: double (nullable = true)
 |-- loss_ratio: double (nullable = true)
 |-- avg_commission: double (nullable = true)
 |-- inforce_prem: double (nullable = true)
 |-- inforce_policies: integer (nullable = true)


── All reporting periods in SR23217A_1